<a href="https://colab.research.google.com/github/AzizAbusaleh/PCCL/blob/main/SAND.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Extracting all data from the `enum_library` table

This cell will connect to the PCCL database, query all data from the `enum_library` table, and then save it to a CSV file.

In [ ]:
import psycopg2
import pandas as pd

# Ensure the db object is initialized (it should be from previous cells)
# db = EnumLibraryDB() # Uncomment and run if db is not defined

try:
    # Establish connection using the parameters from the db object
    with psycopg2.connect(**db.conn_params) as conn:
        sql_query = "SELECT * FROM enum_library;"
        # Use pandas read_sql to directly fetch data into a DataFrame
        full_df = pd.read_sql(sql_query, conn)

    # Save the DataFrame to a CSV file
    output_filename = "enum_library_full_data.csv"
    full_df.to_csv(output_filename, index=False)

    print(f"Successfully extracted {len(full_df)} rows to '{output_filename}'.")
    print("First 5 rows of the extracted data:")
    display(full_df.head())

except Exception as e:
    print(f"An error occurred: {e}")

## Setup and Dependencies

First, we need to install the `sand`, `lance`, and `faiss-gpu` libraries. Since your script uses `torch.cuda.is_available()`, please ensure you are running a GPU runtime (`Runtime > Change runtime type > GPU`).

In [ ]:
git clone https://github.com/pfizer-opensource/SAND.git
cd SAND
uv sync --extra gpu
source .venv/bin/activate
mkdir -p checkpoints
gh release download v1.0.0\
  --repo https://github.com/pfizer-opensource/SAND \
  --pattern "sand_k256_m32.ckpt" \
  --dir checkpoints

# Convert the CSV database to LANCE

In [ ]:
import torch
from sand import Index
from pathlib import Path
import json
import re
import pandas as pd

# --- Define paths for your pre-built assets ---
# IMPORTANT: Replace these with the actual paths to your files
# Example: if you download them to a 'data' folder:
# DATA_DIR = Path('./data')
# CHECKPOINT_PATH = DATA_DIR / 'your_model_checkpoint.pt'
# CONFIG_PATH = DATA_DIR / 'your_config.json'
# LANCE_DB_PATH = DATA_DIR / 'your_lance_dataset.lance'
# FAISS_INDEX_PATH = DATA_DIR / 'your_faiss_index.bin'

# Placeholder paths - YOU MUST UPDATE THESE!
# You might need to create the 'data' directory and place your files there.
# For example, if you clone a GitHub repo containing these files, adjust the paths accordingly.
DATA_DIR = Path('./data') # Create this directory and place your files inside, or adjust as needed
DATA_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = DATA_DIR / 'example_checkpoint.pt'
CONFIG_PATH = DATA_DIR / 'example_config.json'
LANCE_DB_PATH = DATA_DIR / 'example_lance_dataset.lance'
FAISS_INDEX_PATH = DATA_DIR / 'example_faiss_index.bin'

# Ensure these files/directories exist, or create mock ones for demonstration
# In a real scenario, you would download or mount these files.
if not CHECKPOINT_PATH.exists():
    print(f"Warning: Checkpoint file not found at {CHECKPOINT_PATH}. Please provide a valid path.")
    # Example of creating a dummy file for demonstration purposes if needed
    # torch.save({'state_dict': {}}, CHECKPOINT_PATH)

if not CONFIG_PATH.exists():
    print(f"Warning: Config file not found at {CONFIG_PATH}. Please provide a valid path.")
    # Example: create a dummy config
    # with open(CONFIG_PATH, 'w') as f:
    #     json.dump({}, f)

if not LANCE_DB_PATH.exists():
    print(f"Warning: Lance dataset not found at {LANCE_DB_PATH}. Please provide a valid path.")
    # Note: Creating a dummy Lance dataset requires the lance library and a dataframe.
    # For a real run, you must provide a valid Lance path.

if not FAISS_INDEX_PATH.exists():
    print(f"Warning: FAISS index file not found at {FAISS_INDEX_PATH}. Please provide a valid path.")
    # Example: create a dummy file
    # with open(FAISS_INDEX_PATH, 'w') as f:
    #     f.write('')


print(f"Using CHECKPOINT_PATH: {CHECKPOINT_PATH}")
print(f"Using CONFIG_PATH: {CONFIG_PATH}")
print(f"Using LANCE_DB_PATH: {LANCE_DB_PATH}")
print(f"Using FAISS_INDEX_PATH: {FAISS_INDEX_PATH}")

## Helper Function: `slugify`

This function is used to create a clean, URL-friendly string, which is useful for generating filenames.

In [ ]:
def slugify(value: str, fallback: str) -> str:
    slug = re.sub(r"[^A-Za-z0-9._-]+ ", "_", value).strip("._-")
    return slug[:80] or fallback

## Query the SAND Index

Now, we'll set up the user interface using Colab forms for your SMILES query and the number of top hits (`top_k`). The code will then load your pre-built SAND index and execute the query, saving the results to a CSV file and displaying the top entries.

In [ ]:
#@title Enter your query details
query_smiles = 'Cc1ccccc1' #@param {type:"string"}
top_k = 100 #@param {type:"integer"}
n_probe = 512 #@param {type:"integer"}
num_workers = 8 #@param {type:"integer"}
output_dir = 'query_results' #@param {type:"string"}

# Create output directory
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("An H100/CUDA device is required for this script. Please enable GPU runtime.")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

try:
    # Load the SAND Index
    sand_index = Index.from_checkpoint(
        checkpoint_path=str(CHECKPOINT_PATH),
        config_path=str(CONFIG_PATH),
        n_probe=n_probe,
        index_on_gpu=True,
        model_on_gpu=True,
        db_file_path=str(LANCE_DB_PATH),
        index_file_path=str(FAISS_INDEX_PATH),
        compile_encoder=True, # Assuming compilation is desired
        amp_dtype="bf16",
        matmul_precision="high",
        overwrite_index=False, # Set to False to load existing index
        num_workers=num_workers,
    )

    print(f"Running query: {query_smiles} with top_k={top_k}")
    results_df = sand_index(query_smiles, k=top_k)

    # Save results to CSV
    query_slug = slugify(query_smiles, 'query')
    results_filename = output_path / f"{query_slug}_results.csv"
    results_df.to_csv(results_filename, index=False)

    # Save query details to JSON
    query_details_filename = output_path / f"{query_slug}_query.json"
    with open(query_details_filename, 'w') as f:
        json.dump({"query": query_smiles, "top_k": top_k}, f, indent=2)

    print(f"Query successful! Results saved to '{results_filename}'.")
    print(f"Query details saved to '{query_details_filename}'.")

    print("\nTop 5 results:")
    display(results_df.head())

except Exception as e:
    print(f"An error occurred during indexing or querying: {e}")
    print("Please ensure your checkpoint, config, Lance dataset, and FAISS index paths are correct and the files exist.")
    print("Also, verify that the installed 'sand' library and its dependencies are compatible with your pre-built assets.")